# TCGA-LUAD Phase 5: DESeq2 normalization & VST
Run top-to-bottom. This pilot is **exploratory only**, not a tumor-normal differential-expression test. Upload both the Phase 5 project ZIP and your original `tcga_luad_results.zip`.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
project = next((n for n in uploaded if n.endswith('Phase5_Project.zip')), None)
results = next((n for n in uploaded if n == 'tcga_luad_results.zip'), None)
assert project and results, 'Upload TCGA_LUAD_Phase5_Project.zip and tcga_luad_results.zip'
with zipfile.ZipFile(project) as z:
    z.extractall('/content')
root = Path('/content/TCGA_LUAD_Phase5')
assert (root/'preflight.py').exists()


## 1. Validate actual counts and cohort structure
This creates aligned raw-count and metadata files for DESeq2. It does not normalize the data.


In [ ]:
import subprocess
subprocess.run(['python',str(root/'preflight.py'),results,'--output',str(root/'colab_preflight')],check=True)


## 2. Install R and DESeq2
This installation requires internet access and may take several minutes. The Bioconductor version is selected by BiocManager for the R version installed in Colab.


In [ ]:
!apt-get update -qq && apt-get install -y -qq r-base
!Rscript -e 'install.packages("BiocManager", repos="https://cloud.r-project.org"); BiocManager::install(c("DESeq2", "ggplot2", "pheatmap"), ask=FALSE, update=FALSE)'


## 3. DESeq2 median-of-ratios normalization, VST and sensitivity PCA
The R script intentionally does **not** run `DESeq()` or tumor-normal hypothesis testing.


In [ ]:
subprocess.run(['Rscript',str(root/'phase5_deseq2.R'),str(root/'colab_preflight'),str(root/'deseq2_outputs')],check=True)
print('Generated:')
for p in sorted((root/'deseq2_outputs').iterdir()): print(p.name)


## 4. Inspect and download results


In [ ]:
import pandas as pd
from IPython.display import display
sf=pd.read_csv(root/'deseq2_outputs'/'size_factors.csv')
display(sf)
print((root/'deseq2_outputs'/'INTERPRETATION.txt').read_text())
import shutil
out=shutil.make_archive('/content/TCGA_LUAD_Phase5_Results','zip',root_dir=str(root),base_dir='deseq2_outputs')
files.download(out)
